In [1]:
# Checking torch and torchvision version 
import torch 
import torchvision 

print(torch.__version__)
print(torchvision.__version__)

KeyboardInterrupt: 

#### 1. Importing necessary libraries 

In [ ]:
import torch 
import torchvision
import matplotlib.pyplot as plt 
from torch import nn
from torchvision import transforms

# try importing summary from torchinfo 
try: 
    from torchinfo import summary 
except: 
    print("['INFO'] couldn't find ... Downloading it.")
    !pip install -q torchinfo 
    from torchinfo import summary 
    
# try importing our 05_going_modular scripts 
try: 
    from going_modular.going_modular import data_setup, engine 
except: 
    print("\nCouldn't find going_modular ... Downloading from github")
    !git clone "https://github.com/Abisheklimbu/pytorch.git"
    !mv pytorch/going_modular . 
    !rm -rf pytorch
    from  going_modular.going_modular import data_setup, engine 
    

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

#### 2. Get Data

In [ ]:
import os
import zipfile

from pathlib import Path

import requests

# Setup path to data folder
data_path = Path("data/")
image_path = data_path / "pizza_steak_sushi"

# If the image folder doesn't exist, download it and prepare it... 
if image_path.is_dir():
    print(f"{image_path} directory exists.")
else:
    print(f"Did not find {image_path} directory, creating one...")
    image_path.mkdir(parents=True, exist_ok=True)
    
    # Download pizza, steak, sushi data
    with open(data_path / "pizza_steak_sushi.zip", "wb") as f:
        request = requests.get("https://github.com/Abisheklimbu/pytorch/raw/main/data_creation/data/pizza_steak_sushi.zip")
        
        print("Downloading pizza, steak, sushi data...")
        f.write(request.content)

    # Unzip pizza, steak, sushi data
    with zipfile.ZipFile(data_path / "pizza_steak_sushi.zip", "r") as zip_ref:
        print("Unzipping pizza, steak, sushi data...") 
        zip_ref.extractall(image_path)

    # Remove .zip file
    os.remove(data_path / "pizza_steak_sushi.zip")

In [ ]:
train_dir = image_path/"train"
test_dir = image_path/"test"

In [ ]:
print(f"Total training image: {len(list(Path(train_dir).glob("*/*.jpg")))}")
print(f"Total testing image: {len(list(Path(test_dir).glob("*/*.jpg")))}")

#### 3. Create dataset and dataloaders 

In [ ]:
# manual_transforms 
manual_transforms = torchvision.transforms.Compose([
    transforms.Resize((224,224)), 
    transforms.ToTensor(), 
    transforms.Normalize(mean= [0.485, 0.456, 0.406], 
                        std= [0.229, 0.224, 0.225])
])
manual_transforms

In [ ]:
# auto_transforms
weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT
auto_transforms = weights.transforms()
auto_transforms

In [ ]:
train_dataloader, test_dataloader, num_classes = data_setup.create_dataloaders(train_dir=train_dir, 
                                                                              test_dir=test_dir, 
                                                                              transform=auto_transforms, 
                                                                              batch_size=32)

In [ ]:
train_dataloader, test_dataloader, num_classes

#### 4. Load pre-trained models 

In [ ]:
weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT 
model = torchvision.models.efficientnet_b0(weights=weights).to(device)
model

#### 5. Let's look at the summary and freeze features layers 

In [ ]:
summary(model= model, 
        input_size=(32, 3, 224, 224),
        col_names=["input_size", "output_size", "num_params", "trainable"], 
        col_width= 20,
        row_settings= ["var_names"])

In [ ]:
# freeze features layers
for param in model.features.parameters():
    param.requires_grad= False

torch.manual_seed(42)
torch.cuda.manual_seed(42)
model.classifier = nn.Sequential(
    torch.nn.Dropout(p=0.2, inplace=True),
    torch.nn.Linear(in_features=1280, 
                   out_features=len(num_classes), 
                   bias=True).to(device)
)

In [ ]:
# let's again print summary
summary(model=model, 
        input_size=(32,3,224,224), 
        verbose=0, 
        col_names=["input_size", "output_size", "num_params", "trainable"], 
        col_width=20, 
        row_settings=["var_names"]
       )

#### 6. Train model 

In [ ]:
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)
from timeit import default_timer as timer

start_timer = timer()
results = engine.train(model=model, 
                      train_dataloader=train_dataloader, 
                      test_dataloader=test_dataloader, 
                      loss_fn=loss_fn, 
                      optimizer=optimizer,
                      epochs=5, 
                      device=device)
end_timer = timer()
print(f"Total training time {end_timer-start_timer:.3f} seconds")

In [ ]:
results

#### 7. Evaluate model by plotting loss and accuracy 

In [ ]:
try: 
    from helper_function import plot_loss_curves

except: 
    print("Couldn't find. Downloading ...")
    with open("helper_function.py", "wb") as file:
        request = requests.get("https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/helper_functions.py")
        file.write(request.content)
    from helper_function import plot_loss_curves 

In [ ]:
plot_loss_curves(results)

In [ ]:
def plot_loss_curve(results): 
    train_loss = results["train_loss"]
    test_loss = results["test_loss"]

    train_acc = results["train_acc"]
    test_acc = results["test_acc"]

    epochs = range(len(results["train_loss"]))

    plt.figure(figsize=(15,7))

    plt.subplot(1,2,1)
    plt.plot(epochs, train_loss, label="train_loss")
    plt.plot(epochs, test_loss, label="test_loss")
    plt.title("Loss")
    plt.xlabel("Epochs")
    plt.legend()

    plt.subplot(1,2,2)
    plt.plot(epochs, train_acc, label="train_acc")
    plt.plot(epochs, test_acc, label="test_acc")
    plt.title("Accuracy")
    plt.xlabel("Epochs")
    plt.legend()

In [ ]:
plot_loss_curve(results)

#### 8. Predict and plot test images

In [ ]:
from typing import List, Tuple 
from torchvision import transforms 
from PIL import Image 

def pred_and_plot(model:torch.nn.Module, 
                  image_path:str, 
                  class_names:List[str], 
                  image_size:Tuple[int, int]=(224,224),
                  transform:torchvision.transforms = None, 
                  device:torch.device = device):
    
    # load image 
    image = Image.open(image_path)
    
    # create transforms
    if transform is not None: 
        image_transform = transform 
    else:
        image_transform = torchvision.transforms.Compose([
            transforms.Resize((224,224)), 
            transforms.ToTensor(), 
            transforms.Normalize(mean= [0.485, 0.456, 0.406], 
                                 std= [0.229, 0.224, 0.225])
            ])
    
    # model to device 
    model.to(device)
    
    # eval 
    model.eval()
    with torch.inference_mode():
        transformed_image = image_transform(image).unsqueeze(dim=0)
        target_img_pred = model(transformed_image.to(device))
        
    target_img_pred_probs = torch.softmax(target_img_pred, dim=1)
    target_img_pred_label = torch.argmax(target_img_pred_probs, dim=1)
    
    plt.figure()
    plt.imshow(image)
    plt.title(f"Pred: {class_names[target_img_pred_label]} | prob: {target_img_pred_probs.max():.2f}")
    plt.axis("off")

In [ ]:
import random
num_img_to_plot = 3 
test_image_path = list(Path(test_dir).glob("*/*.jpg"))
test_image_random_sample = random.sample(population=test_image_path, 
                                         k=num_img_to_plot)

for image_path in test_image_random_sample:
    pred_and_plot(model=model, 
                  image_path=image_path, 
                  class_names=num_classes,
                  )

#### 9. Predict and plot on cutsom image

In [ ]:
# Download custom image
import requests

# Setup custom image path
custom_image_path = data_path / "04-pizza-dad.jpeg"

# Download the image if it doesn't already exist
if not custom_image_path.is_file():
    with open(custom_image_path, "wb") as f:
        # When downloading from GitHub, need to use the "raw" file link
        request = requests.get("https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/images/04-pizza-dad.jpeg")
        print(f"Downloading {custom_image_path}...")
        f.write(request.content)
else:
    print(f"{custom_image_path} already exists, skipping download.")

# Predict on custom image
pred_and_plot(model=model,
                    image_path=custom_image_path,
                    class_names=num_classes)